In [1]:
from torch.utils.data import Dataset, DataLoader
import torch
import torch.nn as nn
from torch.optim import SGD
from tqdm.notebook import tqdm
from sklearn.model_selection import train_test_split
import pandas as pd
import numpy as np
import networkx as nx
import time
import math
import copy
import os

torch.manual_seed(0)
np.random.seed(0)

In [2]:
from os import chdir
from pathlib import Path

new_path = Path("/Users/haowen/Documents/Decentral RS/fed-learning-main")

if new_path.exists():
    os.chdir(new_path)
    print(f"Working directory changed to: {Path.cwd()}")
else:
    print("Path does not exist.")


Working directory changed to: /Users/haowen/Documents/Decentral RS/fed-learning-main


In [3]:
from src.models.MatrixFactorization import MF, UMF
from src.graphs import random_k_out_graph, create_cycle_graph
from src.users import User
from src.training.decentralized import (decentralized_train_n_epochs,
                                        decentralized_validate_loop)
from src.data_utils import create_batched_dataloaders, create_dataloader

In [4]:
#Make data sample iterable
from torch.utils.data import Dataset, DataLoader, TensorDataset, Sampler
import torch
from tqdm.notebook import tqdm
from sklearn.model_selection import train_test_split
import pandas as pd
import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.optim import SGD

from collections import Counter
import networkx as nx
from networkx.generators.classic import empty_graph
from networkx.utils import discrete_sequence, py_random_state, weighted_choice

import seaborn as sns
from sklearn.utils import shuffle

## Parameter Setting

In [6]:
#{'lr': 0.04518354114581989, 'wd': 0.07432773840871296, 'mom': 0.5104116722654509}
model = "umf"
val_loader_type = "rs"
train_loader_type = "rs"
userprop = None
n_factors = 30
sparse = False
batch_size = 10
lr = 0.04518354114581989
mom = 0.5104116722654509
weight_decay = 0.07432773840871296
graph_seed = 1
n_epochs = 15

## Main

In [8]:
SAMPLED_DATA_DIR       = r"/Users/haowen/Documents/Decentral RS/rebuttal/code/8"


train_path = os.path.join(SAMPLED_DATA_DIR, f"hm_subset_train.csv")
test_path   = os.path.join(SAMPLED_DATA_DIR, f"hm_subset_test.csv")

train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)
train_df.head()

,user_id,item_id,bought
0,850,206,1
1,520,7,1
2,630,2196,1
3,46,1145,1
4,1078,680,1


In [9]:
n_users = train_df['user_id'].nunique()
n_items = train_df['item_id'].nunique()
print(f"Total User: {n_users}")
print(f"Total Item: {n_items}")

Total User: 1760
Total Item: 2881


In [10]:
train_df, val_df = train_test_split(train_df, test_size=0.2, random_state=0)
train_data_loader = create_dataloader(
        df=train_df, dl_type=train_loader_type, batch_size=batch_size, p=train_loader_type
    )
val_data_loader = create_dataloader(df=val_df, dl_type=val_loader_type)
test_data_loader = create_dataloader(df=test_df, dl_type=val_loader_type)


In [11]:
users = {}
for i in tqdm(range(n_users)):
    # model = MF(n_users=n_users, n_items=n_items)
    user_model = UMF(n_items, n_factors = n_factors, sparse = sparse)
    # model = GeneralizedMFOneLayer(n_users=n_users, n_items=n_items)
    #users[i] = User(id=i, model=user_model, optimizer=SGD(user_model.parameters(), lr=lr, weight_decay=weight_decay), model_name = model)
    users[i] = User(
            id=i,
            model=user_model,
            optimizer=SGD(user_model.parameters(), lr=lr, weight_decay=weight_decay, momentum=mom),
            model_name=model,
        )

  0%|          | 0/1760 [00:00<?, ?it/s]

In [12]:
graph = create_cycle_graph(n_users)

In [13]:
train_losses, val_losses, time_per_epoch, commutes = decentralized_train_n_epochs(
        user_models=users,
        train_loader=train_data_loader,
        val_loader=val_data_loader,
        epochs=n_epochs,
        graph=graph,
    )

  0%|          | 0/62568 [00:00<?, ?it/s]

Epoch 1 | Train Loss: 1.8706 | Validation Loss: 1.2764 | Time Elapsed: 38.639195 sec |Commute: 125136 | Commute 
Cost: 5588010648

  0%|          | 0/62568 [00:00<?, ?it/s]

Epoch 2 | Train Loss: 0.3833 | Validation Loss: 0.8018 | Time Elapsed: 39.922986 sec |Commute: 125136 | Commute 
Cost: 5588010648

  0%|          | 0/62568 [00:00<?, ?it/s]

...

Epoch 6 | Train Loss: 0.4117 | Validation Loss: 0.5650 | Time Elapsed: 49.961959 sec |Commute: 125136 | Commute 
Cost: 5588010648

  0%|          | 0/62568 [00:00<?, ?it/s]

Epoch 7 | Train Loss: 0.4159 | Validation Loss: 0.5534 | Time Elapsed: 41.062775 sec |Commute: 125136 | Commute 
Cost: 5588010648

Early stopping.

Total time elapsed: 305.364726875

In [14]:
test_loss = decentralized_validate_loop(users, test_data_loader)

In [15]:
test_loss

0.5357643